# OTP 경로탐색용 OD 데이터 생성
> TCN(Trip Chain Network) → OTP(OpenTripPlanner) 입력용 OD pair 집계

**목적**: 스마트카드 통행 데이터를 OTP 경로탐색 요청에 사용할 수 있는 OD pair 형태로 변환

**출력 컬럼**:
- `od_pair`: OD 키 (승차정류장ID_하차정류장ID)
- `o_stop_id`, `d_stop_id`: 정류장 ID
- `o_lat`, `o_lon`, `d_lat`, `d_lon`: 좌표
- `departure_time`: 대표 출발시간
- `trip_count`: 총 통행 건수 (여러 날짜 합산)

---
## 1. 설정

In [1]:
import pandas as pd

%load_ext autoreload
%autoreload 2

# OTP용 OD pair 그룹화 모듈
from module.tcn_to_otp_od import (
    process_tcn_to_otp_input,
    create_otp_od_data,
    merge_multiple_tcn,
    load_and_merge_tcn_files
)

In [2]:
# TCN 파일 경로 설정
# dates = ['20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223']
dates = ['20250217']

tcn_paths = [f'../data/tcn/{d}/TCN_{d}_route.parquet' for d in dates]

print(f"TCN 파일 수: {len(tcn_paths)}")

TCN 파일 수: 1


---
## 2. OD pair 집계

In [3]:
# OTP 입력용 OD pair 생성
od_data = process_tcn_to_otp_input(
    tcn_paths=tcn_paths,
    output_path='../data/otp/input/otp_od_input.csv'  # 저장 경로
)

od_data.head()

Loading ../data/tcn/20250217/TCN_20250217_route.parquet...
  11,981,768 trips

Merging 1 files...
Total OD pairs: 2,974,640
Total trips: 11,981,768
Saved to ../data/otp/input/otp_od_input.csv


,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10003_1006,10003,1006,37.35292,126.94574,37.515467,126.90765,20250217100320,2
1,10003_10661,10003,10661,37.35292,126.94574,37.463600,126.89756,20250217074728,1
2,10003_10700,10003,10700,37.35292,126.94574,37.452220,126.90157,20250217142818,2
3,10003_10792,10003,10792,37.35292,126.94574,37.457110,126.90559,20250217193916,1
4,10003_109410,10003,109410,37.35292,126.94574,37.476940,126.91546,20250217135920,1


---
## 3. 필터링 및 저장

In [2]:
od_data = pd.read_csv('../data/otp/input/otp_od_input.csv')

In [4]:
# 최소 통행 건수 필터링 (직접 조정)
MIN_TRIPS = 6

od_filtered = od_data[od_data['trip_count'] >= MIN_TRIPS].copy()
od_filtered = od_filtered.reset_index(drop=True)

print(f"필터링 전: {len(od_data):,}")
print(f"필터링 후 (>= {MIN_TRIPS}건): {len(od_filtered):,}")
print(f"제거된 OD pair: {len(od_data) - len(od_filtered):,}")

od_filtered.head()

필터링 전: 2,974,640
필터링 후 (>= 6건): 336,498
제거된 OD pair: 2,638,142


,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10003_8001060,10003,8001060,37.35292,126.94574,37.36000,126.94823,20250217124616,7
1,10003_8001754,10003,8001754,37.35292,126.94574,37.36555,126.94636,20250217124614,10
2,10003_8001755,10003,8001755,37.35292,126.94574,37.36788,126.94520,20250217045519,9
3,10003_8001757,10003,8001757,37.35292,126.94574,37.37393,126.94234,20250217203534,14
4,10003_8001761,10003,8001761,37.35292,126.94574,37.38452,126.93355,20250217142254,13


In [5]:
# 통계 확인
print("=== OD 통계 ===")
print(f"총 OD pair 수: {len(od_filtered):,}")
print(f"총 통행 건수: {od_filtered['trip_count'].sum():,}")
print(f"\n통행 건수 분포:")
print(od_filtered['trip_count'].describe())

# 필터링된 데이터 저장
od_filtered.to_csv('../data/otp/input/otp_od_input_filtered.csv', index=False)
print("\n저장 완료: ../data/otp/input/otp_od_input_filtered.csv")

=== OD 통계 ===
총 OD pair 수: 336,498
총 통행 건수: 7,974,579

통행 건수 분포:
count    336498.000000
mean         23.698741
std          52.337581
min           6.000000
25%           7.000000
50%          11.000000
75%          21.000000
max        2434.000000
Name: trip_count, dtype: float64

저장 완료: ../data/otp/input/otp_od_input_filtered.csv
